In [19]:
pip install settrade-v2

Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

try:
    # Connect to database
    conn = psycopg2.connect(
        host="127.0.0.1", 
        dbname="postgres", 
        user="postgres", 
        password="Shifa.326459"
    )
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()
    
    # Create the database 'advance_api' (Drop if it exists to start fresh)
    cur.execute("DROP DATABASE IF EXISTS advance_api")
    cur.execute("CREATE DATABASE advance_api")
    
    print("[SUCCESS] Database 'advance_api' created successfully.")
    
except Exception as e:
    print(f"[ERROR] Could not create database: {e}")

[SUCCESS] Database 'advance_api' created successfully.


In [21]:
import psycopg2
from settrade_v2 import Investor
from datetime import datetime

# ==========================================
# 1. Configuration
# ==========================================

# Database Credentials
DB_HOST = "127.0.0.1"
DB_NAME = "advance_api"
DB_USER = "postgres"
DB_PASS = "Shifa.326459"

# Settrade Sandbox Credentials
APP_ID = "TDTD3txI7F67RArw"
APP_SECRET = "ZnggI1LfMWalAnDQrTVU726GdgMQO4GWKm7W8UQshHU="
BROKER_ID = "SANDBOX"
APP_CODE = "SANDBOX"

TARGET_SYMBOL = "ADVANC"

# ==========================================
# 2. Connection Setup
# ==========================================

try:
    # 2.1 Connect to Settrade API
    investor = Investor(
        app_id=APP_ID,
        app_secret=APP_SECRET,
        broker_id=BROKER_ID,
        app_code=APP_CODE,
        is_auto_queue=False
    )
    market = investor.MarketData()
    print("[SUCCESS] Connected to Settrade Sandbox.")

    # 2.2 Connect to PostgreSQL (advance_api)
    conn = psycopg2.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)
    conn.set_session(autocommit=True)
    cur = conn.cursor()
    print(f"[SUCCESS] Connected to Database '{DB_NAME}'.")

    # ==========================================
    # 3. Schema Design & Normalization
    # ==========================================

    # Table 1: Master Data (stock_info)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS stock_info (
            symbol VARCHAR(10) PRIMARY KEY,
            company_name VARCHAR(255),
            industry VARCHAR(100),
            sector VARCHAR(100),
            website VARCHAR(255)
        );
    """)

    # Table 2: Transaction Data (daily_prices)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS daily_prices (
            transaction_date DATE NOT NULL,
            symbol VARCHAR(10) NOT NULL,
            open_price NUMERIC,
            high_price NUMERIC,
            low_price NUMERIC,
            close_price NUMERIC,
            volume BIGINT,
            PRIMARY KEY (transaction_date, symbol),
            CONSTRAINT fk_symbol
                FOREIGN KEY(symbol) 
                REFERENCES stock_info(symbol)
        );
    """)
    print("[SUCCESS] Database Schema created successfully.")

    # ==========================================
    # 4. Insert Master Data
    # ==========================================
    
    # Insert static company details for ADVANC
    sql_master = """
        INSERT INTO stock_info (symbol, company_name, industry, sector, website)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (symbol) DO NOTHING;
    """
    cur.execute(sql_master, (
        TARGET_SYMBOL, 
        "Advanced Info Service Public Company Limited", 
        "Technology", 
        "Information & Communication Technology", 
        "http://investor.ais.co.th"
    ))
    print(f"[SUCCESS] Master data for {TARGET_SYMBOL} inserted.")

    # ==========================================
    # 5. Fetch and Insert Transaction Data (10 Years)
    # ==========================================

    print(f"[PROCESSING] Fetching 10-year historical data for {TARGET_SYMBOL}...")
    
    # Calculation: 10 Years * approx 260 trading days = 2600 days
    # Setting limit to 2600 to cover the 10-year period.
    history = market.get_candlestick(
        symbol=TARGET_SYMBOL, 
        interval="1d", 
        limit=1000, 
        normalized=True
    )

    # --- Debugging Section ---
    total_records = len(history['time'])
    print(f"--------------------------------------------------")
    print(f"[DEBUG] API Returned: {total_records} records")
    # Note: If this prints 1 record, it is due to Sandbox limitations.
    print(f"[DEBUG] Raw timestamps: {history['time']}") 
    print(f"--------------------------------------------------")

    count = 0
    # Loop through the data
    for i in range(total_records):
        
        # Convert Unix Timestamp to Date String
        ts = int(history['time'][i])
        date_val = datetime.fromtimestamp(ts).strftime('%Y-%m-%d')
        
        open_val = history['open'][i]
        high_val = history['high'][i]
        low_val = history['low'][i]
        close_val = history['close'][i]
        vol_val = history['volume'][i]
        
        # SQL Upsert (Insert or Update if exists)
        sql_insert = """
            INSERT INTO daily_prices (transaction_date, symbol, open_price, high_price, low_price, close_price, volume)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (transaction_date, symbol) DO UPDATE 
            SET close_price = EXCLUDED.close_price;
        """
        
        cur.execute(sql_insert, (
            date_val,
            TARGET_SYMBOL,
            open_val,
            high_val,
            low_val,
            close_val,
            vol_val
        ))
        count += 1

    print(f"[SUCCESS] Data insertion completed. Total records: {count} days.")

except Exception as e:
    print(f"[ERROR] An error occurred: {e}")

[SUCCESS] Connected to Settrade Sandbox.
[SUCCESS] Connected to Database 'advance_api'.
[SUCCESS] Database Schema created successfully.
[SUCCESS] Master data for ADVANC inserted.
[PROCESSING] Fetching 10-year historical data for ADVANC...
--------------------------------------------------
[DEBUG] API Returned: 729 records
[DEBUG] Raw timestamps: [1675616400, 1675702800, 1675789200, 1675875600, 1675962000, 1676221200, 1676307600, 1676394000, 1676480400, 1676566800, 1676826000, 1676912400, 1676998800, 1677085200, 1677171600, 1677430800, 1677517200, 1677603600, 1677690000, 1677776400, 1678122000, 1678208400, 1678294800, 1678381200, 1678640400, 1678726800, 1678813200, 1678899600, 1678986000, 1679245200, 1679331600, 1679418000, 1679504400, 1679590800, 1679850000, 1679936400, 1680022800, 1680109200, 1680195600, 1680454800, 1680541200, 1680627600, 1680800400, 1681059600, 1681146000, 1681232400, 1681664400, 1681750800, 1681837200, 1681923600, 1682010000, 1682269200, 1682355600, 1682442000, 168

In [22]:
# Table 3: Fundamental Data (เก็บค่า PE, PBV, EPS รายวัน)
cur.execute("""
        CREATE TABLE IF NOT EXISTS daily_fundamentals (
            log_date DATE NOT NULL,
            symbol VARCHAR(10) NOT NULL,
            pe NUMERIC,
            pbv NUMERIC,
            eps NUMERIC,
            div_yield NUMERIC,
            market_cap NUMERIC,
            PRIMARY KEY (log_date, symbol),
            CONSTRAINT fk_fund_symbol
                FOREIGN KEY(symbol) 
                REFERENCES stock_info(symbol)
        );
    """)
print("[SUCCESS] Table 'daily_fundamentals' created.")

[SUCCESS] Table 'daily_fundamentals' created.


In [23]:
# ==========================================
    # 6. Fetch and Insert Fundamental Data (Today's Snapshot)
    # ==========================================
print(f"[PROCESSING] Fetching fundamental data for {TARGET_SYMBOL}...")

    # ดึงข้อมูล Quote (Snapshot ปัจจุบัน)
quote = market.get_quote_symbol(TARGET_SYMBOL)
    
    # ดึงค่าที่ต้องการ (ใช้ .get เพื่อกัน Error กรณีค่าเป็น None)
    # หมายเหตุ: ใน Sandbox ค่าบางตัวอาจจะเป็น 0 หรือ None
pe = quote.get('pe')
pbv = quote.get('pbv')
eps = quote.get('eps')
div_yield = quote.get('percentYield') # ใน dict ชื่อ key คือ percentYield
mkt_cap = quote.get('marketCap')      # บางที Sandbox อาจไม่มี field นี้

    # เตรียมวันที่บันทึก (ใช้วันปัจจุบัน)
today_date = datetime.now().strftime('%Y-%m-%d')
sql_fund = """
        INSERT INTO daily_fundamentals (log_date, symbol, pe, pbv, eps, div_yield, market_cap)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (log_date, symbol) DO UPDATE 
        SET pe=EXCLUDED.pe, 
            pbv=EXCLUDED.pbv, 
            eps=EXCLUDED.eps,
            div_yield=EXCLUDED.div_yield;
    """

cur.execute(sql_fund, (
        today_date, 
        TARGET_SYMBOL, 
        pe, 
        pbv, 
        eps, 
        div_yield, 
        mkt_cap
    ))
    
print(f"[SUCCESS] Fundamental data inserted: PE={pe}, PBV={pbv}, EPS={eps}")

[PROCESSING] Fetching fundamental data for ADVANC...
[SUCCESS] Fundamental data inserted: PE=24.42, PBV=11.23, EPS=11.3


In [26]:
sql = "SELECT * FROM daily_prices"
df = pd.read_sql(sql, conn)

df.head(10)

C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_15420\2090614869.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,transaction_date,symbol,open_price,high_price,low_price,close_price,volume
0,2023-02-06,ADVANC,198.5,198.5,197.0,197.0,2112826
1,2023-02-07,ADVANC,197.0,198.0,196.0,196.5,3223977
2,2023-02-08,ADVANC,196.0,196.5,195.0,195.5,3770659
3,2023-02-09,ADVANC,195.0,201.0,194.0,199.5,11698104
4,2023-02-10,ADVANC,202.0,206.0,201.0,206.0,17037520
5,2023-02-13,ADVANC,208.0,210.0,207.0,210.0,11099764
6,2023-02-14,ADVANC,211.0,211.0,206.0,208.0,8697801
7,2023-02-15,ADVANC,207.0,210.0,206.0,209.0,5608181
8,2023-02-16,ADVANC,209.0,212.0,208.0,211.0,6838734
9,2023-02-17,ADVANC,210.0,211.0,208.0,209.0,6407369


In [7]:
cur.close()
print("[INFO] Cursor closed.")
conn.close()
print("[INFO] Database connection closed.")

[INFO] Cursor closed.
[INFO] Database connection closed.
